In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments,  Trainer, DataCollatorForLanguageModeling
import transformers
import json
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig
import pandas as pd

In [2]:
df = pd.read_excel('../my_benchmarks/CHPC-ollamaRAG/HTMLDOC_Extraction.xlsx')
# apply lambda requires [] around the output. That makes each element in df['text'] a LIST. 
# This format is for conversations features many back and forths. Each row is a list 

df['messages'] = df.apply(lambda x: [ 
        {"role": "user", "content": x["prompt"]},
        {"role": "assistant", "content": x["completion"]}
    ],
    axis=1)
# Unfortunately this dataset is not perfect. One of the lines has content: None which crashes everything
delete_rows = []
for i, row in df.iterrows():
  for d in row['messages']: # This is a list where each element is a dictionary
     for k,v in d.items(): #Dict with keys 
        if pd.isna(v):
           print('None found at row ',i)
           delete_rows.append(i)
print(delete_rows)
df = df.drop(delete_rows, axis=0)
df = df.drop(['prompt','completion'], axis=1)
hf_dataset = Dataset.from_pandas(df)


None found at row  359
[359]


In [3]:
app_repo = "/scratch/general/vast/app-repo/huggingface/"
model_name = "Qwen/Qwen3.5-27B"  #"google/gemma-4-31B-it"  #"Qwen/Qwen3-32B" # "Qwen/Qwen3.5-27B"   "google/gemma-3-27b-it"

tokenizer = AutoTokenizer.from_pretrained(app_repo + model_name)
model = AutoModelForCausalLM.from_pretrained(
  app_repo + model_name,
  dtype=torch.bfloat16,
  device_map="cuda:0",     # I want explicit placement on GPU -  auto  cuda:0
    # attn_implementation="flash_attention_2",
        local_files_only=True
 # quantization_config=bnb_config
)


between tokenizer and model instant


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

In [4]:
# See how many layers are in the model
model

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 5120)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (act): SiLUActivation()
          (conv1d): Conv1d(10240, 10240, kernel_size=(4,), stride=(1,), padding=(3,), groups=10240, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=6144, out_features=5120, bias=False)
          (in_proj_qkv): Linear(in_features=5120, out_features=10240, bias=False)
          (in_proj_z): Linear(in_features=5120, out_features=6144, bias=False)
          (in_proj_b): Linear(in_features=5120, out_features=48, bias=False)
          (in_proj_a): Linear(in_features=5120, out_features=48, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=5120, out_features=17408, bias=False)
          (up_proj): Linear(in_features=5120, out_features=17408, bias=False)
          (down_proj): L

In [5]:
# Gemma 4 is very multimodal focused which is not good for our purposes.
# `mm_token_type_ids` is required as a model input when training it
# target modules is which layers/gates to traintarget_modules = ["q_proj","k_proj","v_proj","o_proj","up_proj","down_proj","gate_proj"]
if 'gemma' in model_name:
    target_modules = r".*\.language_model.*\.(q_proj|k_proj|v_proj|o_proj|up_proj|down_proj|gate_proj|gate_up_proj)" #Gemma 4 is so new the standard modules don't work yet

lora_config = LoraConfig(
    r=16, # good starting point - higher r takes more GPU memory - 8 or 32 also used
    lora_alpha=32,  # baseline is 2 * r
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules
)


In [6]:
# def formatting_prompts_func(example):
#     return [tokenizer.apply_chat_template(msg, tokenize=False) for msg in example["text_jsonized"]]

sftConfig = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_checkpointing=True,  # Enables memory-efficient training
        gradient_accumulation_steps = 8,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        packing=False,
        output_dir = "outputs",
        dataset_text_field="messages",
    #dataset_kwargs={'skip_prepare_dataset': True}
       # max_length=2048,
        # dataset_text_field = "text",
    # chat_template_path should be automatically selected later.
    # completion_only_loss=False   #This means use prompt and completion in training!
)


In [7]:
# Interesting memory saving technique if needed. If you don't care about audio/video multimodal processing, you can remove
# those parts.
#model = AutoModelForCausalLM.from_pretrained(...)
#del base_model.model.vision_tower
#del base_model.model.embed_vision
#del base_model.model.audio_tower
#del base_model.model.embed_audio
#model = get_peft_model(...)

In [9]:
# print(hf_dataset['messages'])
trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    peft_config=lora_config, #Combines model with LORA adapter, do not do this here if you did it elsewhere
    processing_class = tokenizer,
    #tokenizer=tokenizer,
    args=sftConfig,
    #formatting_func=None
)
See how many parameters we are actually training.trainer.model.print_trainable_parameters()

Tokenizing train dataset:   0%|          | 0/3266 [00:00<?, ? examples/s]

[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


trainable params: 79,691,776 || all params: 26,975,690,240 || trainable%: 0.2954


In [ ]:
trainer.train()
#NOTE: trainer.save_model only saves the LORA adapter information.
trainer.save_model("./lora_" + model_name.split("/")[1])
